In [ ]:
#| default_exp window

In [ ]:
#| hide
import socket
from http.server import BaseHTTPRequestHandler, HTTPServer
from importlib.util import find_spec
from threading import Event, Thread
from fastcore.test import *
from nbdev.showdoc import *

A native window over a loopback server, and the menus around it.

pywebview is imported inside the functions that need it, never at module scope. Importing `kavacha.window` on a machine with no window costs nothing, and the parts that decide something run anywhere: the readiness check, the URL wait, the size parsing, the splash, the chord table.

The Dock, the menu bar and the text-substitution defaults are macOS. `menus`, `quiet_text_substitution`, `keep_running_in_dock` and `watch_open_events` check `sys.platform` first and answer `[]`, False or None off darwin. `dock_menu`, `_decorate` and the two Objective-C classes do not check, and are reached only from a caller that does.

Every failure here is printed and swallowed. A missing inline browser, a menu bar that will not build, a window that will not come back: the app runs without them. `run_shell` is the exception, and it raises before it opens anything.

`backend`, `shell_ready` and `wait_for_http` also exist in `probe`. The two `wait_for_http` differ, and the difference is in the section below.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import os, sys, threading, time

In [ ]:
#| export
from fastcore.all import Path

In [ ]:
#| export
from fastcore.xdg import xdg_config_home

In [ ]:
#| export
from kavacha.menus import Binding, Js, MenuItem, Std

In [ ]:
#| export
#: The host kavacha is wearing: `use_app` sets its name, config directory and environment prefix.
APP_NAME = os.environ.get('KAVACHA_APP_NAME') or 'app'
ENV_PREFIX = 'KAVACHA_'

def use_app(name, cfg=None, env_prefix='KAVACHA_'):
    "Wear `name`: its config directory, its storage and its environment prefix."
    global APP_NAME, CFG, STORAGE, ENV_PREFIX, EAGER_CDP
    APP_NAME, ENV_PREFIX = str(name or 'app'), str(env_prefix or '')
    CFG = Path(cfg) if cfg else Path(os.environ.get(f'{ENV_PREFIX}CFG') or xdg_config_home()/APP_NAME)
    STORAGE, EAGER_CDP = CFG/'webview', f'{ENV_PREFIX}CDP_EAGER'
    return APP_NAME

In [ ]:
#| export
CFG = Path(os.environ.get('KAVACHA_CFG') or xdg_config_home()/APP_NAME)

`APP_NAME` and `CFG` are read from the environment once, at import. A host that wants its own name or its own configuration directory sets `$KAVACHA_APP_NAME` or `$KAVACHA_CFG` before importing this module. Setting either afterwards changes nothing.

Reading them creates nothing on disk.

In [ ]:
#| export
def errstr(e): return f"{type(e).__name__}: {e}"

`errstr` names the exception type as well as its message, and every printed failure in this module goes through it. A `KeyError` prints its key and nothing else, an `ImportError` names a module and not what wanted it, and several of these exceptions carry an empty message. The type is what makes the printed line worth reading.

In [ ]:
#| exec_doc
errstr(ValueError('port 9223 is in use')), errstr(KeyError('cocoa'))

In [ ]:
#| export
BACKENDS = {'darwin': 'cocoa', 'win32': 'edgechromium', 'linux': 'gtk'}

In [ ]:
#| export
DEFAULT_SIZE = (1440, 900)

In [ ]:
#| export
MIN_SIZE = (900, 560)

In [ ]:
#| export
STORAGE = CFG/'webview'

In [ ]:
#| export
EAGER_CDP = 'KAVACHA_CDP_EAGER'   #: `use_app` re-spells this

`DEFAULT_SIZE` is the size a window opens at. `MIN_SIZE` is two things: the floor `window_size` enforces on a requested size, and the `min_size` every window is built with, so the size a user can drag a window down to is the size the app is willing to open at.

`STORAGE` is the web engine's own profile: cookies, local storage, and whatever else it keeps between runs. It sits inside `CFG`. `make_window` creates it, and that is the only directory this module creates.

`EAGER_CDP` is the name of the environment variable `warm_cdp` reads, not its value.

In [ ]:
#| exec_doc
DEFAULT_SIZE, MIN_SIZE, str(STORAGE).replace(str(CFG), '/cfg')   # the config directory stands in

In [ ]:
#| export
class ShellApi:
    "What the page may ask the native window for, as `window.pywebview.api.*`."
    #: Set by `run_shell`. Takes a folder and gives it a window of its own.
    on_open_folder = None
    def open_folder(self, path):
        "Open a folder in a window of its own. pywebview runs every `js_api` call on its own thread."
        if not path or self.on_open_folder is None: return False
        self.on_open_folder(str(path))
        return True
    def pick_folder(self):
        "The platform's own folder chooser. A path, or None if it was cancelled."
        import webview
        if (window := webview.active_window()) is None: return None
        try: picked = window.create_file_dialog(webview.FileDialog.FOLDER)
        except Exception as e:
            print(f'  folder dialog: {errstr(e)}')
            return None
        return str(picked[0]) if picked else None
    #: Set by `run_shell`. Gives the page the recent-folder list before the server is up.
    on_recent = None
    def recent(self):
        "Whatever the host offers as recent folders, or nothing."
        return list(self.on_recent()) if self.on_recent else []

`ShellApi` is the whole of what the page may ask the native window for. pywebview exposes each public method as `window.pywebview.api.<name>` and calls it on a thread of its own, so nothing here may assume the main thread.

`on_open_folder` and `on_recent` belong to the host and are set by `run_shell`. Until they are set, `open_folder` returns False and `recent` returns an empty list. An empty path returns False without calling the host.

`open_folder` returns once the host has been called. True says the request was accepted, not that a window appeared: `run_shell` wraps the host's callback so the work leaves the calling thread immediately.

`recent` returns a list, whatever the host's callback returns. The page is given a JSON array either way.

`pick_folder` is the platform's own folder chooser and is the one method that needs pywebview. It returns None when there is no active window, when the chooser was cancelled, and when the chooser raised. Nothing on this page exercises it.

In [ ]:
#| exec_doc
api = ShellApi()
opened = []
api.on_open_folder = opened.append
api.open_folder('/proj/demo'), api.open_folder(''), opened

In [ ]:
#| hide
test_eq(ShellApi().open_folder('/proj/demo'), False)      # no host, so nothing is accepted
test_eq(ShellApi().recent(), [])
api.on_recent = lambda: iter(['/proj/demo', '/proj/notes'])
test_eq(api.recent(), ['/proj/demo', '/proj/notes'])      # an iterator from the host is still a list
api.open_folder(Path('/proj/notes'))
test_eq(opened[-1], '/proj/notes')                        # and a path arrives as a string

In [ ]:
#| export
def backend(platform=None):
    "pywebview's GUI name for `platform`, or None where Leela has no supported webview."
    return BACKENDS.get(platform or sys.platform)

In [ ]:
#| export
def shell_ready(platform=None):
    "`(ok, why)`: whether a native window can be opened here, and what is missing if not."
    gui = backend(platform)
    if gui is None: return False, f'no native webview backend for {platform or sys.platform}'
    try: import webview  # noqa: F401
    except ImportError as e: return False, f'pywebview is not installed ({errstr(e)})'
    return True, gui

A platform absent from `BACKENDS` has no native window here. `shell_ready` answers that case and the missing-pywebview case in one pair. `ok` is False for both and `why` says which it was. When `ok` is True, `why` is the pywebview GUI name, which `run_shell` passes on as `gui`.

Importing `webview` is how `shell_ready` finds out. The `ImportError` is the answer rather than a failure, and this is the only place the module decides whether a window is possible.

In [ ]:
#| exec_doc
backend('darwin'), backend('win32'), backend('plan9')

In [ ]:
#| exec_doc
test_eq(shell_ready('plan9'), (False, 'no native webview backend for plan9'))
ok, why = shell_ready('darwin')
test_eq(ok, why == 'cocoa')   # the gui name when a window can be opened, what is missing when not

In [ ]:
#| export
def wait_for_http(url, timeout=60, interval=.1):
    "Block until `url` answers, or `timeout` passes. True if the server came up."
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=.5).close()
            return True
        except urllib.error.HTTPError: return True   # a 4xx proves the port is live
        except Exception: time.sleep(interval)
    return False

`wait_for_http` polls until the server answers or `timeout` passes. It never raises, and False means the timeout passed.

An error status is an answer. A 4xx proves something is listening on the port, which is the question being asked, so a workspace whose root is not routed yet is not waited out for a minute. `probe.wait_for_http` treats a 404 as a not-yet instead, and the two are not interchangeable.

Every other exception is a not-yet. A connection refused while the server is still binding its port is the ordinary case, not a failure.

In [ ]:
#| hide
#| exec_doc
class _Root(BaseHTTPRequestHandler):
    def do_GET(self): self.send_response(200 if self.path == '/' else 404); self.end_headers()
    def log_message(self, *a): pass
srv = HTTPServer(('127.0.0.1', 0), _Root); Thread(target=srv.serve_forever, daemon=True).start()
url = f'http://127.0.0.1:{srv.server_port}/'
s = socket.socket(); s.bind(('127.0.0.1', 0)); dead = f'http://127.0.0.1:{s.getsockname()[1]}/'; s.close()

In [ ]:
#| exec_doc
wait_for_http(url), wait_for_http(url + 'missing', timeout=.3)   # a 404 proves the port is live

In [ ]:
#| hide
test_eq(wait_for_http(dead, timeout=.3, interval=.05), False)    # nothing is listening at all

In [ ]:
#| export
def start_cdp(port=9223, headless=True, wait=False, timeout=None, profile=None):
    "Bring up the persistent CDP Chrome the inline browser renders into, on a thread."
    def boot():
        try:
            from fossick.cdp import _debug_ready, cdp_setup
            from fossick.core import syncy
            if _debug_ready(port): return
            syncy(cdp_setup(port=port, headless=headless, timeout=timeout, user_data_dir=profile))
        except Exception as e: print(f'  inline browser unavailable: {errstr(e)}')
    t = threading.Thread(target=boot, daemon=True, name='kavacha-cdp-boot')
    t.start()
    if wait: t.join(timeout or 30)
    return t

In [ ]:
#| export
def warm_cdp(port=9223, profile=None):
    "Start Chrome at launch only if `$KAVACHA_CDP_EAGER` asks for it. See `docs/desktop.md`."
    if os.environ.get(EAGER_CDP, '').strip() not in ('1', 'true', 'yes'): return None
    return start_cdp(port, headless=True, profile=profile)

The inline browser is a Chrome the host drives over CDP, brought up by `fossick`. `start_cdp` boots it on a daemon thread and returns that thread. Nothing about the window depends on it. fossick absent, Chrome absent or a boot that fails prints one line and the app carries on without an inline browser. `wait=True` joins the thread, bounded by `timeout`, or by 30 seconds when there is none.

`warm_cdp` is the call made at launch. It starts nothing unless `$KAVACHA_CDP_EAGER` is `1`, `true` or `yes`. The value is stripped before it is compared and the comparison is case-sensitive, so `TRUE` is not one of them. Anything else, including an unset variable, returns None and boots no thread.

In [ ]:
#| hide
if find_spec('fossick') is None:                 # with fossick installed this would start a Chrome
    for v in ('1', 'true', 'yes', ' yes '):
        os.environ[EAGER_CDP] = v
        t = warm_cdp(); assert t is not None, v
        t.join(10); assert not t.is_alive(), 'the boot thread ends rather than hanging'
for v in ('', '0', 'no', 'TRUE'):
    os.environ[EAGER_CDP] = v
    test_is(warm_cdp(), None)
os.environ.pop(EAGER_CDP, None)
test_is(warm_cdp(), None)

In [ ]:
#| export
def window_size(spec=None):
    "`WIDTHxHEIGHT` from `spec` or `$<prefix>WINDOW_SIZE`, else the default. See `use_app`."
    spec = spec or os.environ.get(f'{ENV_PREFIX}WINDOW_SIZE', '')
    try:
        w, h = (int(n) for n in str(spec).lower().split('x', 1))
        if w >= MIN_SIZE[0] and h >= MIN_SIZE[1]: return w, h
    except (ValueError, TypeError): pass
    return DEFAULT_SIZE

`window_size` reads `WIDTHxHEIGHT` from `spec`, or from `$KAVACHA_WINDOW_SIZE` when `spec` is empty. The separator is matched in either case.

A size below `MIN_SIZE` in either dimension is refused, and so is anything that will not parse. Both give `DEFAULT_SIZE`. Nothing raises and nothing warns, so a mistyped variable opens the default window rather than no window.

In [ ]:
#| exec_doc
window_size('1600x1000'), window_size('1280X800'), window_size('320x200'), window_size('wide')

In [ ]:
#| hide
os.environ['KAVACHA_WINDOW_SIZE'] = '1000x700'
test_eq(window_size(), (1000, 700))
test_eq(window_size('1600x1000'), (1600, 1000))    # the argument wins over the environment
os.environ['KAVACHA_WINDOW_SIZE'] = 'huge'
test_eq(window_size(), DEFAULT_SIZE)
del os.environ['KAVACHA_WINDOW_SIZE']
test_eq(window_size(), DEFAULT_SIZE)
test_eq(window_size('900x560'), MIN_SIZE)          # the floor is what it says
test_eq(window_size('900x559'), DEFAULT_SIZE)
test_eq(window_size('1600x1000x900'), DEFAULT_SIZE)
test_eq(window_size((1600, 1000)), DEFAULT_SIZE)   # a pair is not a spec

In [ ]:
#| export
SPLASH = """<!doctype html><meta charset=utf-8><title>{name}</title>
<style>html,body{{height:100%;margin:0;background:#171b20;color:#f4f7f9;
font:15px/1.5 -apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
display:flex;align-items:center;justify-content:center}}
div{{text-align:center;opacity:.85}}b{{display:block;font-size:34px;letter-spacing:-.02em;margin-bottom:10px}}
i{{font-style:normal;font-size:13px;opacity:.6}}</style>
<div><b>{name}</b><i style="{style}">{note}</i></div>"""

def splash(name=None, note='starting the workspace…', style=''):
    "The holding page a window shows before its URL answers, wearing the host's name."
    return SPLASH.format(name=name or APP_NAME, note=note, style=style)

In [ ]:
#| hide
import kavacha.window as _w
test_eq(_w.APP_NAME, 'app')
assert '<b>app</b>' in splash() and '<title>app</title>' in splash()
try:
    use_app('leela', None, 'LEELA_')
    assert '<b>leela</b>' in splash() and '<title>leela</title>' in splash()
    test_eq(_w.EAGER_CDP, 'LEELA_CDP_EAGER')          # read at call time, not captured at import
    test_eq(_w.STORAGE.parent.name, 'leela')
    assert 'did not answer' in _unreachable('http://x')
finally: use_app('app')
test_eq((_w.APP_NAME, _w.EAGER_CDP), ('app', 'KAVACHA_CDP_EAGER'))

In [ ]:
#| export
def _unreachable(url):
    "The same splash, saying why nothing arrived."
    return splash(note=f'the workspace at {url} did not answer', style='color:#e06c75')

The splash is inline HTML referring to nothing outside itself, so a window has something to show before any server exists and asks the network for nothing while it waits.

`_unreachable` is the same page with the waiting line replaced by the address that did not answer, in red. A window whose server never came up says which address failed rather than showing an empty frame.

In [ ]:
#| exec_doc
_unreachable('http://127.0.0.1:8000/').splitlines()[-1]

In [ ]:
#| hide
html = _unreachable('http://127.0.0.1:8000/')
assert 'starting the workspace' not in html, 'the waiting line is replaced, not appended'
assert 'http' not in LOADING, 'the splash fetches nothing'
test_eq(len(html.splitlines()), len(LOADING.splitlines()))

In [ ]:
#| export
DOCK_RECENT = 8          #: rows on the Dock menu, which is a shortcut and not a file browser

In [ ]:
#| export
_dock_target = None

In [ ]:
#| export
def _recent_target(on_folder):
    "One for the life of the app: a menu holds no reference to a Python object, so this must."
    global _dock_target
    if _dock_target is None: _dock_target = _RecentTarget.alloc().init_with(on_folder)
    return _dock_target

In [ ]:
#| export
def dock_menu(on_folder, recent=None):
    "The Dock's own menu: the folders last open, newest first. `recent` is the host's list of paths."
    from AppKit import NSMenu, NSMenuItem
    rows = list(recent or ())[:DOCK_RECENT]
    menu = NSMenu.alloc().init()
    if not rows:
        menu.addItem_(_disabled('No recent folders'))
        return menu
    target = _recent_target(on_folder)
    for path in rows:
        path = str(path)
        item = NSMenuItem.alloc().initWithTitle_action_keyEquivalent_(
            os.path.basename(path.rstrip('/')) or path, b'openRecent:', '')
        item.setToolTip_(path)
        item.setRepresentedObject_(path)
        item.setTarget_(target)
        menu.addItem_(item)
    return menu

In [ ]:
#| export
def _disabled(title):
    from AppKit import NSMenuItem
    item = NSMenuItem.alloc().initWithTitle_action_keyEquivalent_(title, None, '')
    item.setEnabled_(False)
    return item

The Dock menu is the folders last open, at most `DOCK_RECENT` of them, in the order the host gave them. Nothing here sorts the list. The limit is what keeps the menu a shortcut rather than a file browser.

A row is titled by the folder's own name: the basename with a trailing separator removed, and the whole path when that leaves nothing, which is the case for `/`. The full path is both the tooltip and the represented object, and the represented object is what comes back when the row fires.

An empty list still gives a menu. It holds one disabled row, so the Dock menu never looks broken.

`_dock_target` is built once and kept for the life of the process. An `NSMenuItem` holds no strong reference to a Python object, so a target kept nowhere else is collected and its rows do nothing.

Every function here needs AppKit. Nothing on this page exercises them.

In [ ]:
#| export
_hidden, _on_folder, _delegates, _on_recent = [], None, None, None

In [ ]:
#| export
def _dock_delegates():
    "The two delegate subclasses. Built once: an Objective-C class name is global to the runtime."
    global _delegates
    if _delegates is not None: return _delegates
    from Foundation import NO, YES
    from webview.platforms.cocoa import BrowserView
    class Windows(BrowserView.WindowDelegate):
        def windowShouldClose_(self, window):
            window.orderOut_(None)
            if window in _hidden: _hidden.remove(window)
            _hidden.append(window)
            return NO
    class App(BrowserView.AppDelegate):
        def applicationShouldHandleReopen_hasVisibleWindows_(self, app, visible):
            if not visible and _hidden: _hidden.pop().makeKeyAndOrderFront_(None)
            return YES
        def applicationDockMenu_(self, app):
            try: return dock_menu(_on_folder, _on_recent() if _on_recent else None)
            except Exception as e: print(f'  dock menu: {errstr(e)}'); return None
    _delegates = (Windows, App)
    return _delegates

Closing the last window hides it rather than quitting, so the app stays in the Dock. `windowShouldClose_` orders the window out and answers NO. Clicking the Dock icon with no window visible orders the most recently hidden one back in. Cmd+Q still quits.

`_hidden` is a stack, and a window closed twice is moved to the top of it rather than listed twice.

The two delegate classes are built on first use and kept in `_delegates`. An Objective-C class name is global to the runtime, so a second definition of either is an error rather than a second class.

`_on_folder` is module state because `applicationDockMenu_` is handed nothing but the application. `keep_running_in_dock` sets it.

Both subclasses derive from pywebview's cocoa `BrowserView` delegates, so `_dock_delegates` cannot be built on this page.

In [ ]:
#| export
#: What macOS rewrites as you type, and must not while you are typing code. WKWebView hands a
#: `contenteditable` to NSSpellChecker; a browser does not, which is why the packaged app turned
#: quotes into curly quotes and moved the caret out from under CodeMirror, and the served one
#: never did. Inline prediction is in the list because Tab is what accepts one, so the Tab that was
#: meant for the completion list went to the system instead and left the caret at the start of the
#: line. Written to this app's own defaults domain, so nothing outside it changes.
NO_SUBSTITUTION = ('NSAutomaticQuoteSubstitutionEnabled', 'NSAutomaticDashSubstitutionEnabled',
                   'NSAutomaticTextReplacementEnabled', 'NSAutomaticSpellingCorrectionEnabled',
                   'NSAutomaticPeriodSubstitutionEnabled', 'NSAutomaticCapitalizationEnabled',
                   'NSAutomaticInlinePredictionEnabled')

In [ ]:
#| export
def quiet_text_substitution():
    "Turn off the system's autocorrect for this app. True when it took."
    if sys.platform != 'darwin': return False
    try:
        from Foundation import NSUserDefaults
        d = NSUserDefaults.standardUserDefaults()
        for k in NO_SUBSTITUTION: d.setBool_forKey_(False, k)
        return True
    except Exception as e:
        print(f'  the system may still autocorrect what you type: {errstr(e)}')
        return False

`quiet_text_substitution` writes the seven keys to this application's own defaults domain. Nothing outside the application changes.

Off darwin it returns False and writes nothing. A pyobjc that will not import prints one line and returns False. Autocorrect left on is a worse editor, not a broken app, so nothing here raises.

In [ ]:
#| export
def keep_running_in_dock(on_folder=None, recent=None):
    "Hide the last window rather than quit, and fill the Dock menu. Reopen shows one window. Cmd+Q quits."
    global _on_folder, _on_recent
    try:
        from webview.platforms.cocoa import BrowserView
        _on_folder, _on_recent = on_folder, recent
        BrowserView.WindowDelegate, BrowserView.AppDelegate = _dock_delegates()
        return True
    except Exception as e:
        print(f'  the last window will quit {APP_NAME}: {errstr(e)}')
        return False

`keep_running_in_dock` replaces pywebview's two delegate classes. It has an effect only before any window is built, which is why `run_shell` calls it before the first `make_window`.

It returns False and prints when pywebview's cocoa backend cannot be imported. `_on_folder` is set after that import succeeds, so a failed call leaves nothing half-installed.

In [ ]:
#| hide
if sys.platform != 'darwin': test_eq(quiet_text_substitution(), False)
if find_spec('webview') is None:
    test_eq(keep_running_in_dock(print), False)   # it says so, and the app still starts
    test_is(_on_folder, None)                     # nothing was recorded on the way out

In [ ]:
#| export
#: Everything but the title and the size, so a window opened at launch and one opened an hour
#: later are the same window.
WINDOW_KW = dict(min_size=MIN_SIZE, background_color='#171b20', text_select=True, zoomable=True,
                 easy_drag=False)

`WINDOW_KW` is what every window is built with. The title and the size are the two things a caller may vary and are not in it, so a window opened at launch and one opened for a folder an hour later are the same window.

`min_size` is `MIN_SIZE`, the floor `window_size` enforces.

In [ ]:
#| exec_doc
WINDOW_KW

In [ ]:
#| hide
test_is(WINDOW_KW['min_size'], MIN_SIZE)
assert not {'title', 'width', 'height'} & set(WINDOW_KW), 'the per-window arguments stay out of it'

In [ ]:
#| export
_api = None

In [ ]:
#| export
#: Which window is showing which workspace, so a folder already open is raised, not opened twice.
_windows = {}

In [ ]:
#| export
def shell_api():
    "The one `ShellApi` every window shares."
    global _api
    if _api is None: _api = ShellApi()
    return _api

One `ShellApi` is shared by every window. The page reaches it as `window.pywebview.api`, and `run_shell` sets `on_open_folder` on it once, so a window opened later answers a page the same way the first one does.

`_windows` maps a caller's key to the window showing it. `open_window` and `run_shell` are what fill it. A key is whatever the host calls a workspace, and two calls with one key mean one window.

In [ ]:
#| exec_doc
shell_api() is shell_api()

In [ ]:
#| export
def make_window(title=None, size=None):
    "A window on the splash, waiting for a URL."
    import webview
    w, h = size or window_size()
    STORAGE.mkdir(parents=True, exist_ok=True)
    return webview.create_window(title or APP_NAME, html=splash(title or APP_NAME), width=w, height=h,
                                 js_api=shell_api(), **WINDOW_KW)

`make_window` opens on the splash, not on a URL. The window exists at once and `_point` gives it the workspace when the server answers.

The size is `window_size()` unless the caller passes one, so `$KAVACHA_WINDOW_SIZE` reaches every window built without an explicit size. An empty title falls back to `APP_NAME`. `STORAGE` is created here.

It needs pywebview. Nothing on this page opens a window.

In [ ]:
#| export
def off_main_thread(fn, name='kavacha-shell'):
    "Run `fn(arg)` on a thread of its own; `create_window` builds nothing on the main thread."
    def go(arg): threading.Thread(target=fn, args=(arg,), daemon=True, name=name).start()
    return go

`off_main_thread` wraps a one-argument callback so that calling it starts a thread and returns. Every native caller reaches Python on the main thread: a Dock menu row, an Apple Event, a `js_api` call. None of them may build a window there, because the main thread is inside the GUI loop. `run_shell` wraps the host's open-folder callback once and passes the wrapper to everything that can open a folder.

The wrapper returns None. Nothing comes back, and an exception inside `fn` ends its own thread and reaches no caller.

`name` is the thread name, which is what a stack dump shows.

In [ ]:
#| exec_doc
seen, done = [], Event()
go = off_main_thread(lambda p: (seen.append((p, threading.current_thread().name)), done.set()))
go('/proj/demo'); done.wait(5)
seen

In [ ]:
#| export
def open_window(url, title=None, key=None, url_wait=90):
    "A window on `url`, raised rather than opened twice when `key` already has one."
    title = title or APP_NAME
    if key and (window := _windows.get(key)) is not None:
        try: window.show()      # cocoa's `show` is a `callAfter`, so any thread may ask
        except Exception as e: print(f'  could not raise the window for {title}: {errstr(e)}')
        return window
    window = make_window(title)
    if key: _windows[key] = window
    _point(window, url, url_wait)
    return window

In [ ]:
#| export
def _point(window, url, url_wait=90):
    "Wait for the server, then show the workspace. A window that never answers says so."
    if wait_for_http(url, timeout=url_wait): window.load_url(url)
    else: window.load_html(_unreachable(url))

`open_window` with a `key` that already has a window raises that window instead of opening a second one. cocoa answers `show` with a `callAfter`, so any thread may ask. A window that will not come back prints and is returned anyway, because there is no better window to give the caller.

Without a `key` there is no bookkeeping and every call opens a window.

`_point` waits for the server and then loads the URL. A server that has not answered within `url_wait` leaves the window on `_unreachable`, naming the address. `_point` never leaves a window on the splash.

In [ ]:
#| hide
#| exec_doc
class _Win:
    "Stands in for the pywebview window: `open_window` and `_point` use three of its methods."
    def __init__(self, fail=False): self.calls, self.fail = [], fail
    def show(self):
        if self.fail: raise RuntimeError('the window is gone')
        self.calls.append('show')
    def load_url(self, u): self.calls.append(('url', u))
    def load_html(self, h): self.calls.append(('html', h))

In [ ]:
#| exec_doc
_windows['demo'], _windows['gone'] = _Win(), _Win(fail=True)
u = 'http://127.0.0.1:8000/'
open_window(u, 'Demo', key='demo').calls, open_window(u, 'Notes', key='gone').calls

In [ ]:
#| hide
w = _Win(); _point(w, url)
test_eq(w.calls, [('url', url)])
w = _Win(); _point(w, dead, url_wait=.3)
test_eq(w.calls[0][0], 'html')
assert dead in w.calls[0][1], 'the window says which address did not answer'
_windows.clear()

In [ ]:
#| export
def run_shell(urls, titles=(), url_wait=90, gui=None, icon=None, size=None, on_ready=None,
              on_open_folder=None, keys=(), bar=(), lookup=None, on_action=None, on_recent=None):
    "One native window per workspace URL, until they all close. Blocks, on the main thread."
    ok, why = shell_ready()
    if not ok: raise RuntimeError(why)
    import webview
    urls = list(urls)
    if not urls: raise ValueError('the desktop shell needs at least one workspace URL')
    titles = list(titles) + [''] * (len(urls) - len(titles))
    keys = list(keys) + [None] * (len(urls) - len(keys))
    # Every native caller reaches this on the main thread, and none of them may build a window there.
    open_folder = off_main_thread(on_open_folder, 'kavacha-open-folder') if on_open_folder else None
    shell_api().on_open_folder = open_folder
    if sys.platform == 'darwin':
        quiet_text_substitution()                                   # before a web view can ask
        keep_running_in_dock(open_folder, on_recent)                # before any window takes a delegate
    windows = [make_window(t, size) for t in titles]
    for key, window in zip(keys, windows):
        if key: _windows[key] = window
    if open_folder is not None: watch_open_events(open_folder)
    def load(*_):
        "Once the GUI loop is up: wait for the server, then point each window at it."
        for window, url in zip(windows, urls): _point(window, url, url_wait)
        if on_ready is None: return
        try: on_ready()
        except Exception as e: print(f'  desktop shell: {errstr(e)}')
    # One list, shared: pywebview rebuilds the bar whenever the focused window's menu is not the
    # one it built last, and a window opened for a folder later has no menu of its own.
    built = menus(bar, lookup, on_action) if bar else []
    webview.start(load, gui=gui or why, debug=bool(os.environ.get(f'{ENV_PREFIX}WEBVIEW_DEBUG')),
                  private_mode=False, storage_path=str(STORAGE), icon=icon, menu=built)

`run_shell` is the desktop entry point. It opens one window per workspace URL, starts the GUI loop, and blocks until every window is closed. It has to be called on the main thread.

It refuses before it builds anything. A platform with no backend, or a missing pywebview, raises `RuntimeError` carrying the reason `shell_ready` gave. An empty `urls` raises `ValueError`.

`titles` and `keys` are padded to the length of `urls`, so a caller may name the first window and leave the rest.

The order on darwin is fixed. Text substitution is turned off before a web view can ask for it, and the delegates are replaced before any window takes one.

One menu bar list is built and shared by every window. pywebview rebuilds the bar whenever the focused window's menu is not the one it built last, and a window opened for a folder later has no menu of its own.

The URL wait happens inside `load`, which pywebview calls once the GUI loop is up. A server that takes a minute to start still puts a window on screen at once.

In [ ]:
#| hide
if find_spec('webview') is None:   # the readiness check answers before anything imports pywebview
    test_fail(lambda: run_shell(['http://127.0.0.1:8000/']), contains='pywebview is not installed',
              exc=RuntimeError)

In [ ]:
#| export
def menus(bar, lookup, on_action, run=None):
    "The menu bar for `bar`, or an empty one off macOS. Its own function so a test can build one."
    if sys.platform != 'darwin': return []
    try:
        import webview
        webview.settings['SHOW_DEFAULT_MENUS'] = False   # ours are in the order macOS expects
        built, specs = menu_bar(bar, lookup, on_action, run)
        install_menu_patch(specs, lookup)
        return built
    except Exception as e:
        print(f'  no menu bar: {errstr(e)}')
        return []

`menus` returns an empty list off darwin, and an empty list when any part of building the bar fails. An empty list is what `webview.start` takes for no menus, so a bar that cannot be built costs the menus and not the app.

`SHOW_DEFAULT_MENUS` is turned off because `BAR` carries the rows macOS expects, in the order it expects them.

In [ ]:
#| hide
if sys.platform != 'darwin': test_eq(menus(), [])

In [ ]:
#| export
def watch_open_events(on_folder):
    "Answer the `odoc` Apple Event Finder sends a running app. macOS only, best-effort."
    if sys.platform != 'darwin': return None
    try: from Foundation import NSAppleEventManager, NSURL
    except ImportError: return None
    def code(s): return int.from_bytes(s.encode(), 'big')
    def handle(event, _reply):
        try:
            items = event.paramDescriptorForKeyword_(code('----'))
            for i in range(1, (items.numberOfItems() or 0) + 1):
                url = items.descriptorAtIndex_(i).stringValue()
                if not url: continue
                p = NSURL.URLWithString_(url).path() if url.startswith('file:') else url
                if p and os.path.isdir(str(p)): on_folder(str(p))
        except Exception as e: print(f'  open-folder event: {errstr(e)}')
    try:
        mgr = NSAppleEventManager.sharedAppleEventManager()
        mgr.setEventHandler_andSelector_forEventClass_andEventID_(
            _OpenHandler.alloc().init_with(handle), b'handleEvent:reply:',
            code('aevt'), code('odoc'))
        return handle
    except Exception as e:
        print(f'  could not register the open-folder handler: {errstr(e)}')
        return None

Finder sends a running application an `odoc` Apple Event when a folder is dropped on its Dock icon or opened with it. An application that registers no handler is sent the event and drops it, and nothing opens.

`watch_open_events` registers one and returns the handler, so a caller can invoke it directly. It returns None off darwin, when Foundation will not import, and when registration fails.

The handler passes on only paths that are directories on this machine. A `file:` URL is converted through `NSURL`, and anything else is taken as a path already. An event it cannot read prints and is dropped.

In [ ]:
#| hide
if sys.platform != 'darwin': test_is(watch_open_events(print), None)

In [ ]:
#| export
try:                                     # pragma: no cover - macOS only
    import objc
    from Foundation import NSObject
    class _RecentTarget(NSObject):
        "What a Dock menu row fires at. `objc.super`, as `_OpenHandler` needs for the same reason."
        def init_with(self, fn):
            self = objc.super(_RecentTarget, self).init()
            if self is None: return None
            self._fn = fn
            return self
        def openRecent_(self, sender):
            path = sender.representedObject()
            if self._fn and path: self._fn(str(path))

    class _OpenHandler(NSObject):
        "An Apple Event handler must be a selector on an Objective-C object."
        def init_with(self, fn):
            # `objc.super`, not `super()`: the builtin has no `init` to find on an ObjC class, so
            # registration raised and nothing opened a folder dropped on the Dock or picked in Finder.
            self = objc.super(_OpenHandler, self).init()
            if self is None: return None
            self._fn = fn
            return self
        def handleEvent_reply_(self, event, reply): self._fn(event, reply)
except Exception:                        # pragma: no cover - everywhere else
    _OpenHandler = _RecentTarget = None


# The Mac menu bar. pywebview builds it; what is added here is the ordering, the chords it has no
# support for, and the rows AppKit implements itself. See `docs/desktop.md`.

`_RecentTarget` and `_OpenHandler` are Objective-C classes, defined at import where pyobjc is present and bound to None where it is not. The class body runs at import, so any failure inside it, not only a missing pyobjc, leaves both names None. Every caller of either is behind a `sys.platform` check.

In [ ]:
#| hide
if find_spec('objc') is None: test_is(_RecentTarget, None); test_is(_OpenHandler, None)

In [ ]:
#| export
CMD, CTRL, ALT, SHIFT = 1 << 20, 1 << 18, 1 << 19, 1 << 17

In [ ]:
#| export
_MODS = {'mod': CMD, 'cmd': CMD, 'ctrl': CTRL, 'alt': ALT, 'opt': ALT, 'shift': SHIFT,
         'hyper': CMD | CTRL | ALT | SHIFT}

In [ ]:
#| export
#: AppKit spells these as private-use unichars, not as their names.
_NAMED = {'up': '', 'down': '', 'left': '', 'right': '',
          'pageup': '', 'pagedown': '', 'home': '', 'end': '',
          'delete': '', 'enter': '\r', 'return': '\r', 'tab': '\t', 'space': ' ',
          'backspace': '\x08', 'escape': '\x1b'}

`CMD`, `CTRL`, `ALT` and `SHIFT` are AppKit's `NSEventModifierFlag` bits, written out rather than read from AppKit so that a chord can be built on any machine.

`_MODS` is the spelling the host's keymap uses. `mod` and `cmd` are one bit: `mod` is the portable name for the platform's own modifier, and on macOS that is Command. `opt` and `alt` are one bit. `hyper` is all four.

`_NAMED` maps the key names a keymap uses to the single characters AppKit expects. The arrows and the page, home, end and delete keys are private-use characters, which is how AppKit spells them. The rest are control characters.

In [ ]:
#| export
def mac_key(chord):
    "`(keyEquivalent, modifierMask)` for one chord, or None when AppKit cannot spell it."
    parts = str(chord or '').split()[0].split('+') if chord else []
    if not parts or not parts[-1]: return None
    mask, base = 0, parts[-1]
    for p in parts[:-1]:
        if (m := _MODS.get(p.lower())) is None: return None
        mask |= m
    if len(base) == 1 and base.isupper(): mask |= SHIFT
    base = _NAMED.get(base.lower(), base.lower())
    if len(base) != 1: return None
    return base, mask

`mac_key` turns one chord into the pair an `NSMenuItem` wants. None means AppKit cannot spell it, and the row is left with no key equivalent rather than a wrong one.

Only the first stroke is read. A keymap entry written as a sequence, with a space between strokes, has no key equivalent, and its first stroke is what AppKit can install.

A single uppercase base carries its own shift, so `mod+P` and `mod+shift+p` give the same pair. The base is lowercased afterwards: AppKit takes the shift in the modifier mask and the unshifted character as the key.

None comes back for a modifier name that is not in `_MODS`, and for a base that is not one character after `_NAMED` is applied. `f5` is the ordinary case of the second.

In [ ]:
#| exec_doc
mac_key('mod+p'), mac_key('mod+f5'), mac_key('meta+p')

In [ ]:
#| exec_doc
test_eq(mac_key('mod+P'), mac_key('mod+shift+p'))        # an uppercase base carries the shift itself
test_eq(mac_key('hyper+k'), ('k', CMD | CTRL | ALT | SHIFT))
test_eq(mac_key('mod+k mod+s'), mac_key('mod+k'))        # a sequence installs its first stroke

In [ ]:
#| hide
test_eq(mac_key('mod+tab'), ('\t', CMD))
test_eq(mac_key('escape'), ('\x1b', 0))                  # a chord may have no modifier at all
test_eq(len(mac_key('mod+up')[0]), 1)                    # an arrow is one private-use character
for bad in (None, '', 'mod+', 'mod+f5', 'meta+p', 'mod+ctrl+f12'): test_is(mac_key(bad), None)

In [ ]:
#| export
def menu_chord(action, lookup):
    """The chord to put on a menu row, or None to leave it bare.

    A key equivalent is consulted before the keystroke reaches the web view, so anything installed
    here is taken away from the page. Only a global action with a modifier is safe: a scoped one
    would fire outside its scope, and a bare letter would fire while you were typing it.
    """
    k = lookup(action)
    if k is None or not k.bound or k.scope != 'global': return None
    first = k.keys[0]
    if not (set(first.split('+')[:-1]) & {'mod', 'cmd', 'ctrl', 'alt', 'opt', 'hyper'}): return None
    return mac_key(first)

A key equivalent is consulted before the keystroke reaches the web view, so every chord installed on a menu row is taken away from the page. `menu_chord` is the filter that decides which may be taken.

Three things have to hold. The action is bound, its scope is `global`, and its first stroke carries a modifier. A scoped action would fire outside its scope. A bare letter would fire while you were typing it.

`lookup` is the host's keymap, as a callable from an action to a [`Binding`](menus.html) or `None`. kavacha ships no keymap of its own, so a plain dict's `get` is a lookup and is what the examples here use.

In [ ]:
#| export
def _title(row, lookup):
    "The label for an action row: the keymap's, or the action with its underscores opened up."
    k = lookup(row.action)
    t = (k.label if k and k.label else row.action.replace('_', ' '))
    return t[:1].upper() + t[1:]

In [ ]:
#| exec_doc
KEYS = {'save':         Binding(('mod+s',), 'Save'),
        'quick_open':   Binding(('mod+shift+o',)),
        'find_in_file': Binding(('mod+f',), 'Find', scope='editor'),
        'comment':      Binding(('/',), 'Comment'),
        'orphan':       Binding()}
{a: menu_chord(a, KEYS.get) for a in ('save', 'quick_open', 'find_in_file', 'comment', 'orphan')}

`save` is global, bound and modified, so it earns its chord. The other three are each refused for a different reason, and an action the keymap has never heard of is refused too.

In [ ]:
#| hide
test_eq(menu_chord('save', KEYS.get), ('s', CMD))
test_eq(menu_chord('quick_open', KEYS.get), ('o', CMD | SHIFT))
test_is(menu_chord('find_in_file', KEYS.get), None)   # scoped: it would fire outside its scope
test_is(menu_chord('comment', KEYS.get), None)        # bare: it would fire while you typed it
test_is(menu_chord('orphan', KEYS.get), None)         # unbound
test_is(menu_chord('never_heard_of_it', KEYS.get), None)

`_title` prefers the keymap's label. Without one it opens up the action's underscores, so a row is never blank and never shows a bare identifier.

In [ ]:
#| exec_doc
_title(MenuItem('save'), KEYS.get), _title(MenuItem('quick_open'), KEYS.get)

In [ ]:
#| hide
test_eq(_title(MenuItem('save'), KEYS.get), 'Save')          # the label
test_eq(_title(MenuItem('quick_open'), KEYS.get), 'Quick open')
test_eq(_title(MenuItem('never_seen'), KEYS.get), 'Never seen')

In [ ]:
#| export
def menu_bar(bar, lookup, on_action, run=None):
    "pywebview menus for `bar`, plus the table `_decorate` needs to finish them off AppKit-side."
    from webview.menu import Menu, MenuAction, MenuSeparator
    run = run or _menu_run
    menus, specs = [], {}
    for title, rows in bar:
        items = []
        for row in rows:
            if isinstance(row, MenuItem) and not row.action: items.append(MenuSeparator()); continue
            if isinstance(row, Std):
                specs[(title, row.title)] = row
                items.append(MenuAction(row.title, lambda: None))
            elif isinstance(row, Js):
                items.append(MenuAction(row.title, (lambda e: lambda: run(e))(row.expr)))
            else:
                name = _title(row, lookup)
                specs[(title, name)] = row
                items.append(MenuAction(name, (lambda a: lambda: on_action(a))(row.action)))
        menus.append(Menu(title, items))
    return menus, specs

`menu_bar` walks `bar`, the host's menu table, and returns pywebview `Menu` objects together with the table `_decorate` needs afterwards. That table is keyed by `(menu title, row title)`, because a title is all AppKit gives back when the bar is rebuilt.

There are three kinds of row. A `MenuItem` with no action is a separator. A `Js` row runs its expression through `run`. Anything else calls `on_action` with the action name, under the label the keymap gives it.

A `Std` row is one AppKit implements itself, so the callback built here does nothing. `_decorate` gives the row a nil target and the real selector once the bar exists.

`on_action` is where the host says what an action means. kavacha builds no JavaScript: a library that packages somebody else's app does not know the name of their dispatch function. `run` is separate because a `Js` row is JavaScript by definition, and defaults to `_menu_run` against the focused window.

In [ ]:
#| export
def eval_js(js):
    "Run one expression in the window that has focus, and return what it evaluated to."
    import webview
    if (window := webview.active_window()) is None: return None
    try: return window.evaluate_js(js)
    except Exception as e:
        print(f'  menu: {errstr(e)}')
        return None

_menu_run = eval_js

In [ ]:
#| export
_menu_patched = False

In [ ]:
#| export
def install_menu_patch(specs, lookup):
    """Finish each item off as AppKit needs it, wherever pywebview builds the bar.

    It rebuilds on every focus change to a window whose menu differs, so anything done once to the
    bar after start is thrown away. This wraps the one method both rebuild paths go through.
    """
    global _menu_patched
    if _menu_patched: return False
    try:
        from webview.platforms.cocoa import BrowserView
    except Exception as e:
        print(f'  no menu bar: {errstr(e)}')
        return False
    original = BrowserView._recreate_menus
    def recreate(self, user_menu):
        main = original(self, user_menu)
        try: _decorate(main, specs, lookup)
        except Exception as e: print(f'  menu bar: {errstr(e)}')
        return main
    BrowserView._recreate_menus = recreate
    _menu_patched = True
    return True

pywebview rebuilds the whole menu bar whenever the focused window's menu differs from the one it built last, so anything done to the bar once, after start, is thrown away at the next focus change. `install_menu_patch` wraps `_recreate_menus`, the one method both rebuild paths go through, and every rebuilt bar is decorated again.

It patches once. A second call returns False and changes nothing. So does a call where pywebview's cocoa backend will not import.

`specs` is captured by the wrapper, so the table `menu_bar` returned is the table every later rebuild uses.

In [ ]:
#| hide
if find_spec('webview') is None:
    test_eq(install_menu_patch({}, KEYS.get), False)
    test_eq(_menu_patched, False)   # nothing was wrapped, so a later call may still patch

In [ ]:
#| export
def _decorate(main, specs, lookup):
    "Chords onto the rows that may have one, and the responder chain onto the rows AppKit owns."
    import AppKit
    for i in range(main.numberOfItems()):
        sub = main.itemAtIndex_(i).submenu()
        if sub is None: continue
        title = str(sub.title())
        for j in range(sub.numberOfItems()):
            item = sub.itemAtIndex_(j)
            spec = specs.get((title, str(item.title())))
            if spec is None: continue
            if isinstance(spec, Std):
                item.setTarget_(None)          # nil target is what walks the responder chain
                item.setAction_(spec.selector)
                got = mac_key(f'{spec.mods}+{spec.key}') if spec.key else None
            else: got = menu_chord(spec.action, lookup)
            if got: item.setKeyEquivalent_(got[0]); item.setKeyEquivalentModifierMask_(got[1])
        if title == 'Window': AppKit.NSApp().setWindowsMenu_(sub)

`_decorate` finishes the rows AppKit has to finish. A `Std` row is given a nil target and its selector, which is what walks the responder chain and lets Cut, Copy, Paste and Minimize reach the web view. Every other row is given the chord `menu_chord` allows it. A row with no entry in `specs` is left alone.

The Window menu is handed to `NSApp` as the windows menu, which is what puts the open windows on it.

`_decorate` runs inside the wrapper `install_menu_patch` installed, and every exception it raises is caught there. A bar that cannot be decorated is a bar without chords, not a failed launch.

In [ ]:
#| hide
srv.shutdown()